# Plot images

Query img files by plate and site, then download TIFF, combine channels, and visualize. If running this from a newly cloned repository, you must first run 0_prepare_data/1A_download_metadata.py and 0_prepare_data/2A_format_metadata.py and 0_prepare_data/2E_format_image_index.py. 

In [14]:
import polars as pl
import numpy as np
import os
import matplotlib.pyplot as plt
from PIL import Image
from sh import aws
from pathlib import Path

In [15]:
index_path = "../../1_snakemake/inputs/images/index.parquet"
meta_path = "../../1_snakemake/inputs/metadata/metadata.parquet"
tiff_dir = "../../1_snakemake/inputs/images/tiff"
png_dir = "../../1_snakemake/inputs/images/png"
aws_img_path = "s3://cellpainting-gallery/cpg0037-oasis/axiom/images"

Path(tiff_dir).mkdir(parents=True, exist_ok=True)
Path(png_dir).mkdir(parents=True, exist_ok=True)

index = pl.read_parquet(index_path)
meta = pl.read_parquet(meta_path)

index = index.join(meta, on=["Metadata_Plate", "Metadata_Well"])

In [4]:
import numpy as np

def normalize(channel):
    print(channel.min())
    print(channel.max())
    return (channel - channel.min()) / (channel.max() - channel.min())

def normalize_quantile(channel):
    p99 = np.percentile(channel, 99.9)
    print(p99)
    return channel / p99

def normalize_channel(channel, channel_nm):
    if channel_nm == "DNA":
        p99 = 11000
    elif channel_nm == "ER":
        p99 = 13000
    elif channel_nm == "AGP":
        p99 = 8000
    elif channel_nm == "RNA":
        p99 = 7000
    elif channel_nm == "Mito":
        p99 = 11000

    return channel / p99

In [19]:
def get_tiffs(plate: str, well: str, site: int, batch: str, png_dir: str) -> list:
    channels = ["DNA", "ER", "AGP", "RNA", "Mito"]
    img_path = f"{tiff_dir}/{batch}/{plate}"
    tiffs = []

    for channel in channels:
        img_nm = (plot.filter(pl.col("Metadata_Plate") == plate)
                    .filter(pl.col("Metadata_Well") == well)
                    .filter(pl.col("Metadata_Site") == site)
                    .filter(pl.col("Channel") == channel)
        ).select("Filename").item()
        img_nm = os.path.basename(img_nm)

        tiff_path = f"{img_path}/{img_nm}"
        print(tiff_path)

        if not os.path.exists(tiff_path):
            print(f"Downloading: {img_nm}")
            aws_img = f"{aws_img_path}/{batch}/images/{plate}/{img_nm}"
            aws("s3", "cp", aws_img, tiff_path, "--no-sign-request")

        img = np.array(Image.open(tiff_path))
        tiffs.append(normalize_channel(img, channel))   

    return tiffs

In [6]:
from matplotlib.gridspec import GridSpec

def plot_tiffs(tiffs: list, png_path: str) -> None:
    rgb_image = np.zeros((tiffs[0].shape[0], tiffs[0].shape[1], 3))

    rgb_image[:, :, 0] = tiffs[4] + tiffs[2] + tiffs[3]  # Red component (for red, yellow, and purple)
    rgb_image[:, :, 1] = tiffs[1] + tiffs[2]  # Green component (for green and yellow)
    rgb_image[:, :, 2] = tiffs[0] + tiffs[3]  # Blue component (for blue and purple)

    rgb_image = np.clip(rgb_image, 0, 1)

    # Create a grid layout with GridSpec
    fig = plt.figure(figsize=(10, 12))
    gs = GridSpec(2, 5, height_ratios=[1, 4], hspace=0.01, wspace=0.01)

    # First row: Horizontal strip of smaller channel images
    channels = ["DNA", "ER", "AGP", "RNA", "Mito"]

    for i, (channel, title) in enumerate(zip(tiffs, channels)):
        ax = fig.add_subplot(gs[0, i])  # Place in the first row, ith column
        ax.imshow(channel, cmap="gray")
        ax.set_title(title, fontsize=10)
        ax.axis("off")

    # Second row: Combined RGB image
    ax_combined = fig.add_subplot(gs[1, :])  # Span all columns in the second row
    ax_combined.imshow(rgb_image)
    ax_combined.axis("off")

    plt.subplots_adjust(left=0.02, right=0.98, top=0.98, bottom=0.02)

    # Save the figure
    plt.savefig(png_path, dpi=300, bbox_inches="tight", pad_inches=0)
    plt.show()

## AR agonists and antagonists

In [7]:
strong_ag = ["OASIS1137", "OASIS1302", "OASIS1280", "OASIS1381", "OASIS1509", "OASIS1313", "OASIS1933"]
strong_antag = ["OASIS419", "OASIS1502", "OASIS940", "OASIS1098", "OASIS379", "OASIS1488", "OASIS794"]

In [9]:
ag_meta = meta.filter(pl.col("Metadata_OASIS_ID").is_in(strong_ag)).filter(pl.col("Metadata_Concentration") == 100)
antag_meta = meta.filter(pl.col("Metadata_OASIS_ID").is_in(strong_antag)).filter(pl.col("Metadata_Concentration") == 100)

In [ ]:
# From this, define image metadata
plate = "plate_41002897"
batch = "prod_27"
well = "M01"
site = 6

# The get_tiffs function will look for this object
plot = (index.filter(pl.col("Metadata_Plate") == plate)
             .filter(pl.col("Metadata_Well") == well))

png_path = f"{png_dir}/{plate}_{well}_{site}.png"
tiffs = get_tiffs(plate, well, site, batch, tiff_dir)
plot_tiffs(tiffs, png_path)

In [ ]:
for row in antag_meta.iter_rows(named=True): 
    well = row["Metadata_Well"]
    plate = row["Metadata_Plate"]
    batch = row["Metadata_source"]
    batch = batch.replace("assayworks_", "")
    exp = row["Metadata_Compound"]

    site = 6

    plot = (index.filter(pl.col("Metadata_Plate").str.contains(plate))
            .filter(pl.col("Metadata_Well") == well))

    png_path = f"{png_dir}/AR_antagonist/{plate}_{well}_{site}_{exp}.png"

    tiffs = get_tiffs(plate, well, site, batch, tiff_dir)
    plot_tiffs(tiffs, png_path)

## Cell count increases

In [10]:
cmpds = ["Baloxavir marboxil", "Ceritinib", "Alectinib (Hydrochloride)"]
concs = [100, 20]

cc_meta = meta.filter(pl.col("Metadata_Compound").is_in(cmpds)).filter(pl.col("Metadata_Concentration").is_in(concs))

In [ ]:
for row in cc_meta.iter_rows(named=True):
    well = row["Metadata_Well"]
    plate = row["Metadata_Plate"]
    batch = row["Metadata_source"]
    batch = batch.replace("assayworks_", "")
    exp = row["Metadata_Compound"]

    site = 6

    plot = (index.filter(pl.col("Metadata_Plate").str.contains(plate))
            .filter(pl.col("Metadata_Well") == well))

    png_path = f"{png_dir}/cc_increase/{plate}_{well}_{site}_{exp}.png"

    tiffs = get_tiffs(plate, well, site, batch, tiff_dir)
    plot_tiffs(tiffs, png_path)

## Alectinib (crazy phenotype of firework nuclei)

In [ ]:
alectinib_meta = meta.filter(pl.col("Metadata_Compound") == "Alectinib (Hydrochloride)")

for row in alectinib_meta.iter_rows(named=True):
    well = row["Metadata_Well"]
    plate = row["Metadata_Plate"]
    batch = row["Metadata_source"]
    batch = batch.replace("assayworks_", "")
    exp = row["Metadata_Compound"]
    conc = row["Metadata_Concentration"]
    conc = float(f"{conc:.1g}")

    site = 6

    plot = (index.filter(pl.col("Metadata_Plate").str.contains(plate))
            .filter(pl.col("Metadata_Well") == well))

    png_path = f"{png_dir}/alectinib/{plate}_{well}_{site}_{conc}_{exp}.png"

    tiffs = get_tiffs(plate, well, site, batch, tiff_dir)
    plot_tiffs(tiffs, png_path)

We see the fragmented nuclei at both of the highest dose (20.0) and one of the two second-highest (7.0). A few very small DNA fragments at the third highest (2.0).

In [6]:
cmpd_meta = meta.filter(pl.col("Metadata_Compound") == "Colchicine")

for row in cmpd_meta.iter_rows(named=True):
    well = row["Metadata_Well"]
    plate = row["Metadata_Plate"]
    batch = row["Metadata_source"]
    batch = batch.replace("assayworks_", "")
    exp = row["Metadata_Compound"]
    conc = row["Metadata_Concentration"]
    conc = float(f"{conc:.1g}")

    site = 6

    plot = (index.filter(pl.col("Metadata_Plate").str.contains(plate))
            .filter(pl.col("Metadata_Well") == well))

    png_path = f"{png_dir}/colchicine/{exp}_{conc}_{well}_{plate}_{site}.png"

    tiffs = get_tiffs(plate, well, site, batch, tiff_dir)
    plot_tiffs(tiffs, png_path)

## Few DMSO examples

In [22]:
dmso_meta = meta.filter(pl.col("Metadata_Compound") == "DMSO").sample(n=10, seed=1)

In [ ]:
for row in dmso_meta.iter_rows(named=True):
    well = row["Metadata_Well"]
    plate = row["Metadata_Plate"]
    batch = row["Metadata_source"]
    batch = batch.replace("assayworks_", "")
    exp = row["Metadata_Compound"]

    site = 6

    plot = (index.filter(pl.col("Metadata_Plate").str.contains(plate))
            .filter(pl.col("Metadata_Well") == well))

    png_path = f"{png_dir}/dmso/{plate}_{well}_{site}_{exp}.png"

    tiffs = get_tiffs(plate, well, site, batch, tiff_dir)
    plot_tiffs(tiffs, png_path)